# Code 10.1.2: State visitation under two policies

Counts how often each gridworld cell is visited under the uniform random policy and under a policy that takes the optimal action (computed by value iteration, Code 10.2.3) with probability 0.8. The `GridWorld` class is listed in Code 10.2.1.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
import sys
from pathlib import Path

def _add_gridworld_path():
    """The chapter figure scripts ship the gridworld module this listing imports."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / "chapters/10-reinforcement-learning-basics/figures/src/gridworld.py"
        if candidate.is_file():
            folder = str(candidate.parent)
            if folder not in sys.path:
                sys.path.insert(0, folder)
            return
    raise FileNotFoundError(
        "Could not find chapters/10-reinforcement-learning-basics/figures/src/gridworld.py. "
        "Run this notebook from the repository (or a subdirectory of it)."
    )

_add_gridworld_path()


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 10.1.1
import numpy as np


In [ ]:
from gridworld import GridWorld, value_iteration, START

def visit_counts(grid, policy_probs, episodes, rng, max_steps=200):
    counts = np.zeros(grid.n_states)
    for _ in range(episodes):
        s = grid.index[START]
        for _ in range(max_steps):
            counts[s] += 1
            a = rng.choice(grid.n_actions, p=policy_probs[s])
            s, r, done = grid.step(s, a)
            if done:
                counts[s] += 1
                break
    return counts / episodes

grid = GridWorld()
_, greedy = value_iteration(grid, gamma=0.9)
uniform = np.full((grid.n_states, grid.n_actions), 0.25)
eps = 0.2                                   # mostly greedy, sometimes random
mostly_greedy = np.full((grid.n_states, grid.n_actions), eps / 4)
mostly_greedy[np.arange(grid.n_states), greedy] += 1 - eps

rng = np.random.default_rng(1)
c_uniform = visit_counts(grid, uniform, 2000, rng)
c_greedy = visit_counts(grid, mostly_greedy, 2000, rng)
for name, c in [("uniform", c_uniform), ("mostly greedy", c_greedy)]:
    goal = c[grid.index[(0, 4)]]
    print(f"{name:13s}: {c.sum():5.1f} visits per episode, "
          f"reaches goal in {100 * goal:.0f}% of episodes")


**Expected output**

Output:

```text
uniform      :  56.9 visits per episode, reaches goal in 18% of episodes
mostly greedy:  11.0 visits per episode, reaches goal in 99% of episodes
```
